In [1]:
%pip install datasets transformers torch accelerate

In [2]:
# Datasets
from datasets import load_dataset, concatenate_datasets

# Output
import json
from collections import Counter
import pandas as pd

# Model
import torch
import re
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import time

In [3]:
# Load dataset
dataset = load_dataset("gsm8k", "main", split="test")

# Filter datasets
dataset = dataset.filter(
    lambda x: len(x["answer"]) > 404
)

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Dataset({
    features: ['question', 'answer'],
    num_rows: 245
})


In [4]:
# Load model
model_name = "Qwen/Qwen2-1.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [5]:
# Helper function
def extract_true_answer(answer):
    return answer.split("####")[-1].strip()

def extract_true_reasoning(answer):
    return answer.split("####")[0].strip()

def extract_predicted_answer(text):
    # take last number in output
    numbers = re.findall(r"-?\d+\.?\d*", text)
    return numbers[-1] if numbers else None

def count_tokens(input_text, output_text):
    input_tokens = len(tokenizer.encode(input_text))
    output_tokens = len(tokenizer.encode(output_text))
    return input_tokens, output_tokens

In [6]:
def build_prompt(question):
    return f"""You are a helpful math tutor.
Solve step by step and give the final answer.
Provide reasoning and intermediate steps and return the final numerical answer.
Output the final answer in the format on a new line: #### [final numeric answer]

Rules:
- Only provide reasoning and the final answer.
- Do not include any other text or explanations.
- Do not output code for reasoning, the reasoning must be a step by step process in natural language.
- Do not use , or . for formatting of large numbers (e.g. write 1000000 instead of 1,000,000).
- You must follow the exact format for the final answer.
- You must start directly with "Step 1:".
- Final line must be: [final numeric answer]

Response Format:
Step x: [reasoning step]
Step x + 1: [reasoning step]
...
Final Answer:[final numeric answer]

Question: {question}
Answer:"""

In [7]:
pipe = pipeline(
    "text-generation",
    model=model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device=0 if torch.cuda.is_available() else -1
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [8]:
results = []

start_time = time.time()

num_attempts = 2
consistent_count = 0

for i, example in enumerate(dataset):
    question = example["question"]

    true_answer = extract_true_answer(example["answer"])
    true_reasoning = extract_true_reasoning(example["answer"])

    prompt = build_prompt(
        question=question,
    )

    model_responses = []
    model_answers = []
    model_correct = []
    model_input_tokens = []
    model_output_tokens = []
    model_reasonings = []

    # Run 2 attempts (temp=0.7)
    for attempt in range(num_attempts):

        output = pipe(
            prompt,
            max_new_tokens=320,
            return_full_text=False,
            do_sample=True,
            temperature=0.7,
            repetition_penalty=1.1
        )[0]

        response = output["generated_text"]

        model_reasoning = output
        model_answer = extract_predicted_answer(response)


        is_correct = (model_answer == true_answer)

        input_tokens, output_tokens = count_tokens(prompt, response)

        # store per attempt
        model_answers.append(model_answer)
        model_correct.append(is_correct)
        model_input_tokens.append(input_tokens)
        model_output_tokens.append(output_tokens)
        model_reasonings.append(model_reasoning)
        model_responses.append(response)

    response_1 = model_responses[0]
    response_2 = model_responses[1]

    model_reasoning = model_reasonings[0]

    # Consistency (between the 2 attempts)
    cleaned_answers = [ans for ans in model_answers]

    is_consistent = (
        len(set(cleaned_answers)) == 1
        and all(ans != "" for ans in cleaned_answers)
    )

    if is_consistent:
        consistent_count += 1

    # Average metrics
    avg_accuracy = sum(model_correct) / num_attempts
    avg_input_tokens = sum(model_input_tokens) / num_attempts
    avg_output_tokens = sum(model_output_tokens) / num_attempts

    results.append({
        "question": question,
        "true_reasoning": true_reasoning,
        "true_answer": true_answer,
        "model_reasoning": model_reasoning,

        # per attempt
        "model_answers": model_answers,
        "model_correct": model_correct,
        "model_input_tokens": model_input_tokens,
        "model_output_tokens": model_output_tokens,

        # aggregated
        "avg_accuracy": avg_accuracy,
        "avg_input_tokens": avg_input_tokens,
        "avg_output_tokens": avg_output_tokens,
        "consistent": is_consistent,
        "response_1": response_1,
        "response_2": response_2
    })

end_time = time.time()

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'repetition_penalty', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=320) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=320) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/trans

In [9]:
df = pd.DataFrame(results)
# df.to_csv("qwen_gsm8k_results.csv", index=False)

df["true_answer"] = pd.to_numeric(df["true_answer"], errors="coerce")

df.to_json("results.json", orient="records", indent=4)

In [10]:
# Metrics

total_runtime = end_time - start_time
avg_runtime = total_runtime / len(results)
print(f"Total runtime for {len(results)} examples: {total_runtime:.3f} sec")
print(f"Average runtime per example: {avg_runtime:.3f} sec")

accuracy = sum(r["avg_accuracy"] for r in results) / len(results)
print("Final answer accuracy:", accuracy)

avg_input_tokens = sum(r["avg_input_tokens"] for r in results) / len(results)
avg_output_tokens = sum(r["avg_output_tokens"] for r in results) / len(results)

print("Avg input tokens:", avg_input_tokens)
print("Avg output tokens:", avg_output_tokens)

total_questions = len(results)
consistency_score = consistent_count / total_questions if total_questions > 0 else 0
print(f"Consistency score: {consistency_score:.4f}")

Total runtime for 245 examples: 5763.995 sec
Average runtime per example: 23.527 sec
Final answer accuracy: 0.09591836734693877
Avg input tokens: 274.0979591836735
Avg output tokens: 274.9428571428571
Consistency score: 0.0980
